In [1]:
#Setup that works on Colab and on a local machine. Run this cell first.
#Local: run `pip install -r requirements.txt` once, and start the notebook from the repo root.
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    #A fresh Colab VM is empty, so get the repo. Safe to re-run: it only clones if the repo isn't there yet.
    if not Path("Data").exists() and not Path("DL_FinalProject").exists():
        subprocess.run(["git", "clone", "https://github.com/mpchouinard/DL_FinalProject.git"], check=True)
    if Path("DL_FinalProject").exists():
        os.chdir("DL_FinalProject")
    #Colab has torch and scikit-learn, but its pandas is too old to read our pickle, and it lacks transformers.
    #Don't install requirements.txt here, it can replace Colab's GPU build of torch.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pandas", "transformers"], check=True)
    if "pandas" in sys.modules:
        print("pandas was already loaded. Restart the session once (Runtime > Restart session), then run again.")

if not Path("Data/c_elegans_preprocessed.pkl").exists():
    raise FileNotFoundError(f"Start the notebook from the repo root. Current folder: {os.getcwd()}")
print("on Colab:", ON_COLAB, "| working directory:", os.getcwd())

on Colab: False | working directory: c:\Repos\DL_FinalProject


In [ ]:
import copy
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

from src.embed import load_embeddings

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open(Path("Data/c_elegans_preprocessed.pkl"), "rb") as f:
    data = pickle.load(f)
folds = data["folds"]

In [ ]:
import torch
from torch import nn


class CrossAttentionModel(nn.Module):
    """Predict whether a receptor-peptide pair is an agonist.

    Inputs are cached ESM-2 per-residue embeddings with shape
    (batch, padded_sequence_length, embedding_dim), plus true lengths.
    The 650M embeddings use embedding_dim=1280.
    """

    def __init__(self, embedding_dim=1280, num_heads=8, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.receptor_norm = nn.LayerNorm(embedding_dim)
        self.peptide_norm = nn.LayerNorm(embedding_dim)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attended_norm = nn.LayerNorm(embedding_dim)
        self.attended_dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(3 * embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    @staticmethod
    def masked_mean(values, padding_mask):
        # padding_mask is True where a row is padding.
        valid = (~padding_mask).unsqueeze(-1)
        return (values * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1)

    def forward(self, receptor_residues, peptide_residues, receptor_lengths, peptide_lengths):
        # Cached embeddings are float16; match the model's parameter dtype.
        dtype = self.cross_attention.in_proj_weight.dtype
        receptor_residues = receptor_residues.to(dtype=dtype)
        peptide_residues = peptide_residues.to(dtype=dtype)
        receptor_lengths = torch.as_tensor(receptor_lengths, device=receptor_residues.device)
        peptide_lengths = torch.as_tensor(peptide_lengths, device=peptide_residues.device)

        receptor_padding = (
            torch.arange(receptor_residues.size(1), device=receptor_residues.device)[None, :]
            >= receptor_lengths[:, None]
        )
        peptide_padding = (
            torch.arange(peptide_residues.size(1), device=peptide_residues.device)[None, :]
            >= peptide_lengths[:, None]
        )

        receptor = self.receptor_norm(receptor_residues)
        peptide = self.peptide_norm(peptide_residues)

        # Each peptide residue queries the receptor residues. Padded receptor
        # positions cannot receive attention. Padded peptide positions are
        # excluded when we summarize the attended sequence below.
        attended, _ = self.cross_attention(
            query=peptide,
            key=receptor,
            value=receptor,
            key_padding_mask=receptor_padding,
            need_weights=False,
        )
        attended = self.attended_norm(peptide + self.attended_dropout(attended))

        pair_features = torch.cat(
            [
                self.masked_mean(peptide, peptide_padding),
                self.masked_mean(receptor, receptor_padding),
                self.masked_mean(attended, peptide_padding),
            ],
            dim=-1,
        )
        # One raw logit per pair; the training code will choose the loss.
        return self.classifier(pair_features).squeeze(-1)

In [ ]:
#Encoder: turns names into per-residue embeddings plus true lengths.
#The training loop only calls encoder(kind, names). When we add LoRA to ESM-2, we write a new encoder class with the
#same methods that runs ESM-2 on the sequences instead of looking them up.


class CachedEncoder:
    """Looks up precomputed (frozen) ESM-2 embeddings by name."""

    def __init__(self, tag="650M", device=DEVICE, folder="embeddings"):
        self.device = device
        self.tables = {}
        for kind, file_prefix in [("receptor", "receptors"), ("peptide", "peptides")]:
            loaded = load_embeddings(f"{folder}/{file_prefix}_{tag}.npz")
            self.tables[kind] = {
                "row": {name: i for i, name in enumerate(loaded["names"])},
                "lengths": torch.as_tensor(loaded["lengths"], dtype=torch.long, device=device),
                "residues": torch.as_tensor(loaded["residues"], device=device),  # float16, zero padded
            }

    def __call__(self, kind, names):
        table = self.tables[kind]
        rows = torch.as_tensor([table["row"][name] for name in names], device=self.device)
        lengths = table["lengths"][rows]
        longest = int(lengths.max())  # drop padding columns this batch doesn't need
        return table["residues"][rows, :longest], lengths

    #The methods below do nothing for cached embeddings. They are needed for LoRA (adapters to train and save).
    def trainable_parameters(self):
        return []

    def train(self, mode=True):
        return self

    def state_dict(self):
        return {}

    def load_state_dict(self, state):
        pass

In [ ]:
#Batching, losses and evaluation


def iterate_batches(pairs, batch_size, shuffle, generator=None):
    """Yields (receptor names, peptide names, labels) from a pairs dataframe."""
    
    receptors = pairs["GPCR name"].to_numpy()
    peptides = pairs["Peptide"].to_numpy()
    labels = pairs["binds"].to_numpy().astype(np.float32)

    if shuffle:
        order = torch.randperm(len(pairs), generator=generator).numpy()
    else:
        order = np.arange(len(pairs))

    for start in range(0, len(pairs), batch_size):
        batch = order[start:start + batch_size]
        yield list(receptors[batch]), list(peptides[batch]), torch.as_tensor(labels[batch])


def forward_batch(model, encoder, receptor_names, peptide_names):
    receptor_residues, receptor_lengths = encoder("receptor", receptor_names)
    peptide_residues, peptide_lengths = encoder("peptide", peptide_names)
    return model(receptor_residues, peptide_residues, receptor_lengths, peptide_lengths)


def weighted_bce_loss(train_labels, pos_weight="auto"):
    """BCE where agonists count pos_weight times more. "auto" = negatives / positives in the training set."""

    if pos_weight == "auto":
        n_pos = train_labels.sum()
        pos_weight = (len(train_labels) - n_pos) / n_pos
    pos_weight = torch.tensor(float(pos_weight), device=DEVICE)

    def loss_fn(logits, targets):
        return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight)

    return loss_fn


def focal_loss(gamma=2.0, alpha=0.25):
    """BCE that down-weights easy pairs by (1 - p_true)^gamma. alpha=None turns off the class weighting."""

    def loss_fn(logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_true = torch.exp(-bce)
        weight = (1 - p_true) ** gamma
        if alpha is not None:
            weight = weight * (alpha * targets + (1 - alpha) * (1 - targets))
        return (weight * bce).mean()

    return loss_fn


def make_loss(loss_config, train_labels):
    if loss_config["name"] == "bce":
        return weighted_bce_loss(train_labels, loss_config.get("pos_weight", "auto"))
    if loss_config["name"] == "focal":
        return focal_loss(loss_config.get("gamma", 2.0), loss_config.get("alpha", 0.25))
    raise ValueError(f"unknown loss {loss_config['name']}")


@torch.no_grad()
def evaluate(model, encoder, pairs, batch_size=256):
    """AUPRC (main metric, only ~2% of pairs are agonists) and AUROC for a pairs dataframe."""

    model.eval()
    encoder.train(False)

    scores = []
    for receptor_names, peptide_names, _ in iterate_batches(pairs, batch_size, shuffle=False):
        scores.append(forward_batch(model, encoder, receptor_names, peptide_names).float().cpu())
    scores = torch.cat(scores).numpy()

    labels = pairs["binds"].to_numpy().astype(int)
    return {
        "auprc": float(average_precision_score(labels, scores)),
        "auroc": float(roc_auc_score(labels, scores)),
        "n_agonists": int(labels.sum()),
    }

In [ ]:
#Training loop


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_optimizer(model, encoder, config):
    """The head and the encoder get their own learning rates (the encoder has nothing to train until LoRA)."""
    
    groups = [{"params": list(model.parameters()), "lr": config["lr"]}]
    encoder_params = list(encoder.trainable_parameters())
    if encoder_params:
        groups.append({"params": encoder_params, "lr": config.get("encoder_lr", config["lr"])})
    return torch.optim.AdamW(groups, weight_decay=config["weight_decay"])


def train_one_epoch(model, encoder, optimizer, loss_fn, train_pairs, config, generator):
    """One pass over the training pairs. Returns the average loss."""
    model.train()
    encoder.train(True)
    trainable = [p for group in optimizer.param_groups for p in group["params"]]

    total_loss, n_batches = 0.0, 0
    for receptor_names, peptide_names, labels in iterate_batches(
            train_pairs, config["batch_size"], shuffle=True, generator=generator):
        optimizer.zero_grad()
        logits = forward_batch(model, encoder, receptor_names, peptide_names)
        loss = loss_fn(logits, labels.to(DEVICE))
        loss.backward()
        if config.get("grad_clip"):
            torch.nn.utils.clip_grad_norm_(trainable, config["grad_clip"])
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches


def snapshot(model, encoder):
    """Copy of everything trainable (the head now, LoRA adapters later), to restore the best epoch."""

    return {"model": copy.deepcopy(model.state_dict()), "encoder": copy.deepcopy(encoder.state_dict())}


def restore(model, encoder, state):
    model.load_state_dict(state["model"])
    encoder.load_state_dict(state["encoder"])


def train_one_fold(fold, config, encoder, seed=0):
    """Trains on the fold's training pairs and keeps the epoch with the best validation AUPRC.
    Returns the model with those weights loaded. The test set is not used here."""

    set_seed(seed)
    generator = torch.Generator().manual_seed(seed)
    train_pairs = fold["train"]
    val_pairs = fold["hyperparameter_tuning"]

    model = config["model_factory"]().to(DEVICE)
    optimizer = build_optimizer(model, encoder, config)
    loss_fn = make_loss(config["loss"], train_pairs["binds"].to_numpy())

    best_auprc, best_epoch, best_state = -1.0, -1, None
    history = []
    for epoch in range(config["epochs"]):
        train_loss = train_one_epoch(model, encoder, optimizer, loss_fn, train_pairs, config, generator)
        val = evaluate(model, encoder, val_pairs)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_auprc": val["auprc"], "val_auroc": val["auroc"]})
        print(f"  epoch {epoch:2d}  train loss {train_loss:.4f}  val AUPRC {val['auprc']:.4f}  val AUROC {val['auroc']:.4f}")

        if val["auprc"] > best_auprc:
            best_auprc, best_epoch, best_state = val["auprc"], epoch, snapshot(model, encoder)
        elif epoch - best_epoch >= config["patience"]:
            print(f"  early stop (best epoch {best_epoch})")
            break

    restore(model, encoder, best_state)
    return model, {"best_epoch": best_epoch, "best_val_auprc": best_auprc, "history": history, "state": best_state}


def run_all_folds(folds, config, encoder, run_name, fold_ids=None, seed=0):
    """Trains each fold, scores its test fold once with the best epoch, and saves a checkpoint per fold.
    Evaluation on other species is done separately, from the saved checkpoints."""

    checkpoint_folder = Path("checkpoints") / run_name
    checkpoint_folder.mkdir(parents=True, exist_ok=True)
    saved_config = {key: value for key, value in config.items() if not callable(value)}  # lambdas can't be saved

    rows = []
    for fold_id, fold in enumerate(folds):
        if fold_ids is not None and fold_id not in fold_ids:
            continue
        print(f"fold {fold_id}: test families {fold['test_families']}")
        model, info = train_one_fold(fold, config, encoder, seed=seed)
        test = evaluate(model, encoder, fold["test"])
        print(f"  -> test AUPRC {test['auprc']:.4f}  AUROC {test['auroc']:.4f}")

        rows.append({
            "fold": fold_id,
            "best_epoch": info["best_epoch"],
            "val_auprc": info["best_val_auprc"],
            "test_auprc": test["auprc"],
            "test_auroc": test["auroc"],
            "test_agonists": test["n_agonists"],
        })
        torch.save({"state": info["state"], "config": saved_config, "fold": fold_id}, checkpoint_folder / f"fold{fold_id}.pt")
    return pd.DataFrame(rows)

In [ ]:
#One experiment = one config. Change the model, loss or learning rate here and re-run.
config = {
    "model_factory": lambda: CrossAttentionModel(embedding_dim=1280),  # swap in the pooled model here
    "loss": {"name": "bce", "pos_weight": "auto"},                       # or {"name": "focal", "gamma": 2.0, "alpha": 0.25}
    "lr": 1e-4,
    "weight_decay": 0.01,
    "batch_size": 32,
    "epochs": 30,
    "patience": 5,       # stop when validation AUPRC hasn't improved for this many epochs
    "grad_clip": 1.0,
}

encoder = CachedEncoder("650M")  # reads embeddings/ relative to the repo root

results = run_all_folds(folds, config, encoder, run_name="cross_attention_bce")
print(results.to_string(index=False))
print(results[["test_auprc", "test_auroc"]].agg(["mean", "std"]))